In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
## 1. Setup Paths & Configuration

import sys
import os
from pathlib import Path

# Add source directory to path
current_path = Path.cwd()
base_project_dir = current_path.parent
src_dir = base_project_dir / "src"

if str(src_dir) not in sys.path:
    sys.path.append(str(src_dir))

# Define key directories
dataset_directory = src_dir / "abstractionsshapecoder" / "dataset"
saved_directory = src_dir / "abstractionsshapecoder" / "saved"

In [4]:
import pickle
import os
from tqdm import tqdm
from abstractionsshapecoder.shape_parser import ShapeParser
from abstractionsshapecoder.dsl_utils import collect_singleton_and_pair_data

# --- Configuration ---
INPUT_FILE = src_dir / "abstractionsshapecoder" / "prog_data" / "PN_chair.txt"
PICKLE_FILE = saved_directory / "all_dsl_shapes.pkl"
CHAIR_LIMIT = None  # Set to an integer for testing, or None for all

def load_dsl_dataset():
    full_dsl_shapes = {}

    # FIX: Ensure the parent directory exists
    PICKLE_FILE.parent.mkdir(parents=True, exist_ok=True)

    # Check for cached Pickle
    if PICKLE_FILE.exists():
        print(f"[INFO] Loading cached dataset from {PICKLE_FILE}...")
        with open(PICKLE_FILE, "rb") as f:
            full_dsl_shapes = pickle.load(f)
    else:
        print(f"[INFO] Cache not found. Parsing from {INPUT_FILE}...")
        if not INPUT_FILE.exists():
            print(f"[ERROR] Input file not found: {INPUT_FILE}")
            return {}

        parser = ShapeParser()
        with open(INPUT_FILE, 'r') as f:
            lines = f.readlines()

        for line in tqdm(lines, desc="Parsing Shapes"):
            line = line.strip()
            if not line: continue
            try:
                shape_id, prog_text = line.split(':', 1)
                dsl_obj = parser.parse(prog_text)
                if dsl_obj:
                    # Calculate Params immediately
                    s_data, p_data = collect_singleton_and_pair_data([dsl_obj])
                    full_dsl_shapes[shape_id.strip()] = {
                        "dsl": dsl_obj,
                        "singleton_params": s_data,
                        "pair_params": p_data
                    }
            except Exception:
                pass

        # Save to Pickle
        with open(PICKLE_FILE, "wb") as f:
            pickle.dump(full_dsl_shapes, f)

    # Apply Limit
    if CHAIR_LIMIT is not None and len(full_dsl_shapes) > CHAIR_LIMIT:
        print(f"[INFO] Limiting to {CHAIR_LIMIT} shapes.")
        return dict(list(full_dsl_shapes.items())[:CHAIR_LIMIT])
    
    return full_dsl_shapes

shapes_l0 = load_dsl_dataset()
print(f"[RESULT] Loaded {len(shapes_l0)} L0 shapes.")

[INFO] Loading cached dataset from c:\Users\Amogh\abstraction-discovery\src\abstractionsshapecoder\saved\all_dsl_shapes.pkl...
[RESULT] Loaded 1100 L0 shapes.


In [5]:
shapes_l0["172_0_0"]

{'dsl': <abstractionsshapecoder.dsl_nodes.Union at 0x158af28eb70>,
 'singleton_params': {'Cuboid': [[0.68, 0.09, 0.66],
   [0.06, 0.6, 0.06],
   [0.06, 0.6, 0.06],
   [0.06, 0.6, 0.06],
   [0.06, 0.6, 0.06],
   [0.08, 0.79, 0.08],
   [0.08, 0.79, 0.08],
   [0.68, 0.14, 0.24],
   [0.08, 0.79, 0.08],
   [0.08, 0.79, 0.08],
   [0.08, 0.79, 0.08]],
  'Translate': [[-0.0, -0.17, 0.0],
   [-0.31, -0.51, -0.17],
   [-0.31, -0.51, 0.3],
   [0.31, -0.51, -0.17],
   [0.31, -0.51, 0.3],
   [-0.26, 0.27, -0.18],
   [-0.14, 0.27, -0.25],
   [0.0, 0.74, -0.22],
   [-0.0, 0.27, -0.25],
   [0.14, 0.27, -0.25],
   [0.28, 0.27, -0.25]]},
 'pair_params': {'Translate(Cuboid)': [[-0.0, -0.17, 0.0, 0.68, 0.09, 0.66],
   [-0.31, -0.51, -0.17, 0.06, 0.6, 0.06],
   [-0.31, -0.51, 0.3, 0.06, 0.6, 0.06],
   [0.31, -0.51, -0.17, 0.06, 0.6, 0.06],
   [0.31, -0.51, 0.3, 0.06, 0.6, 0.06],
   [-0.26, 0.27, -0.18, 0.08, 0.79, 0.08],
   [-0.14, 0.27, -0.25, 0.08, 0.79, 0.08],
   [0.0, 0.74, -0.22, 0.68, 0.14, 0.24],
  

In [6]:
shapes_l0["172_0_0"]["dsl"]

In [7]:
print(shapes_l0["172_0_0"]["dsl"])

Union(
    Translate(vec=[-0.000, -0.170, 0.000])
        Cuboid(size=[0.680, 0.090, 0.660]),
    Union(
        Translate(vec=[-0.310, -0.510, -0.170])
            Cuboid(size=[0.060, 0.600, 0.060]),
        Union(
            Translate(vec=[-0.310, -0.510, 0.300])
                Cuboid(size=[0.060, 0.600, 0.060]),
            Union(
                Translate(vec=[0.310, -0.510, -0.170])
                    Cuboid(size=[0.060, 0.600, 0.060]),
                Union(
                    Translate(vec=[0.310, -0.510, 0.300])
                        Cuboid(size=[0.060, 0.600, 0.060]),
                    Union(
                        Translate(vec=[-0.260, 0.270, -0.180])
                            Cuboid(size=[0.080, 0.790, 0.080]),
                        Union(
                            Translate(vec=[-0.140, 0.270, -0.250])
                                Cuboid(size=[0.080, 0.790, 0.080]),
                            Union(
                                Translate(vec=[0.000, 0

In [8]:
class SymRef:
    """Mirror symmetry (Reflection) across an axis."""
    def __init__(self, child, axis='AX'):
        self.child = child
        self.axis = axis  # 'AX', 'AY', or 'AZ'

class SymTrans:
    """Translational symmetry (Repetition)."""
    def __init__(self, child, axis, count, distance):
        self.child = child
        self.axis = axis      # 'AX', 'AY', or 'AZ'
        self.count = count    # Integer: number of repetitions
        self.distance = distance  # Float: spacing between repetitions

In [9]:
import numpy as np
import copy
import logging
from abstractionsshapecoder.dsl_nodes import Union, Translate, Cuboid, Rotate

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger("ShapeOptimizer")

class ShapeOptimizer:
    def __init__(self, threshold=0.01):
        self.threshold = threshold

    def is_near(self, a, b):
        """Helper for scalar comparison (abs:0.01)."""
        return abs(float(a) - float(b)) <= self.threshold

    def is_near_vec(self, v1, v2):
        """Helper for vector comparison."""
        return np.all(np.abs(np.array(v1) - np.array(v2)) <= self.threshold)

    def is_zero(self, val):
        """Rule: zero_add or zero_mov check."""
        if isinstance(val, (list, np.ndarray)):
            return np.all(np.abs(np.array(val)) <= self.threshold)
        return abs(float(val)) <= self.threshold

    def apply_optimization_rules(self, node):
        """
        Implements SEM_3DL_REWRITES logic at the individual node level.
        """
        if node is None: return None

        # --- RULE: s3dl_move_comb & s3dl_zero_mov ---
        if isinstance(node, Translate):
            node.child = self.optimize_node(node.child)
            
            # Combine Move(Move) -> Move(Add(a,x))
            if isinstance(node.child, Translate):
                logger.debug("Rule: s3dl_move_comb applied.")
                new_vec = (np.array(node.vector) + np.array(node.child.vector)).tolist()
                return Translate(node.child.child, new_vec)
            
            # Remove No-op Move
            if self.is_zero(node.vector):
                logger.debug("Rule: s3dl_zero_mov applied.")
                return node.child

        # --- RULE: s3dl_rot_comb & s3dl_zero_rot ---
        if isinstance(node, Rotate):
            node.child = self.optimize_node(node.child)
            
            # Combine Rotate(Rotate) if axis is same
            if isinstance(node.child, Rotate) and node.axis == node.child.axis:
                logger.debug("Rule: s3dl_rot_comb applied.")
                return Rotate(node.child.child, node.axis, node.angle + node.child.angle)
            
            # Remove No-op Rotate
            if self.is_zero(node.angle):
                logger.debug("Rule: s3dl_zero_rot applied.")
                return node.child

        # --- RULE: s3dl_rot_over_move ---
        # (Move (Rotate X A v) a b c) -> (Rotate (Move X a b c) A v)
        if isinstance(node, Translate) and isinstance(node.child, Rotate):
            logger.debug("Rule: s3dl_rot_over_move applied.")
            rot_node = node.child
            inner_move = Translate(rot_node.child, node.vector)
            return Rotate(inner_move, rot_node.axis, rot_node.angle)

        # --- RULE: s3dl_union_over_sym ---
        # (SymRef (Union X Y) A) -> (Union (SymRef X A) (SymRef Y A))
        if isinstance(node, SymRef) and isinstance(node.child, Union):
            logger.debug("Rule: s3dl_union_over_sym applied.")
            left_sym = SymRef(node.child.left, node.axis)
            right_sym = SymRef(node.child.right, node.axis)
            return Union(left_sym, right_sym)

        return node

    def find_symmetries(self, parts):
        optimized = []
        used_indices = set()

        for i in range(len(parts)):
            if i in used_indices: continue
            p1 = parts[i]
            
            # --- 1. Translational Symmetry (SymTrans) ---
            # Look for 3+ items with equal spacing along an axis
            for axis_idx, axis_name in enumerate(['AX', 'AY', 'AZ']):
                sequence = [i]
                if not isinstance(p1, Translate): continue
                
                # Find all potential matches in this axis
                for j in range(i + 1, len(parts)):
                    if j in used_indices: continue
                    p2 = parts[j]
                    if isinstance(p2, Translate) and self.is_near_node_geometry(p1.child, p2.child):
                        # Check if it aligns on the current axis but matches others
                        diff = np.array(p1.vector) - np.array(p2.vector)
                        other_axes = [0, 1, 2]
                        other_axes.remove(axis_idx)
                        
                        # Must be aligned on other axes
                        if all(abs(diff[ax]) < self.threshold for ax in other_axes):
                            sequence.append(j)
                
                if len(sequence) >= 3:
                    # Sort by the active axis to find consistent distance
                    sequence.sort(key=lambda idx: parts[idx].vector[axis_idx])
                    vecs = [np.array(parts[idx].vector)[axis_idx] for idx in sequence]
                    dist = vecs[1] - vecs[0]
                    
                    # Verify spacing is consistent
                    is_consistent = all(self.is_near(vecs[k+1] - vecs[k], dist) for k in range(len(vecs)-1))
                    
                    if is_consistent:
                        logger.info(f"Rule: SymTrans detected ({len(sequence)} parts, axis {axis_name})")
                        # Representative is the first item
                        optimized.append(SymTrans(parts[sequence[0]], axis_name, len(sequence), dist))
                        used_indices.update(sequence)
                        break

            if i in used_indices: continue

            # --- 2. Mirror Symmetry (SymRef) ---
            found_mirror = False
            for j in range(i + 1, len(parts)):
                if j in used_indices: continue
                p2 = parts[j]
                
                if isinstance(p1, Translate) and isinstance(p2, Translate):
                    v1, v2 = np.array(p1.vector), np.array(p2.vector)
                    for axis_name, idx in [('AX', 0), ('AY', 1), ('AZ', 2)]:
                        mirrored_v2 = v2.copy()
                        mirrored_v2[idx] = -mirrored_v2[idx]
                        
                        if self.is_near_vec(v1, mirrored_v2) and self.is_near_node_geometry(p1.child, p2.child):
                            optimized.append(SymRef(p1, axis_name))
                            used_indices.add(i); used_indices.add(j)
                            found_mirror = True
                            break
                    if found_mirror: break
            
            if not found_mirror:
                optimized.append(p1)
                used_indices.add(i)
                
        return optimized

    def is_near_node_geometry(self, n1, n2):
        """Checks if two nodes are identical in shape/size (for symmetry)."""
        if type(n1) != type(n2): return False
        if isinstance(n1, Cuboid):
            return self.is_near_vec(n1.size, n2.size)
        # Add more geometry checks (Sphere, Cylinder) here
        return False

    def flatten_unions(self, node):
        if isinstance(node, Union):
            return self.flatten_unions(node.left) + self.flatten_unions(node.right)
        return [node] if node is not None else []

    def rebuild_union(self, nodes):
        if not nodes: return None
        if len(nodes) == 1: return nodes[0]
        # s3dl_union_over ensures we build a consistent tree
        return Union(nodes[0], self.rebuild_union(nodes[1:]))

    def optimize_node(self, node):
        return self.apply_optimization_rules(node)

    def run(self, root_node):
        if root_node is None: return None

        # 1. Deep apply local rules (combining moves, rotations, etc.)
        root_node = self.optimize_node(root_node)
        
        # 2. Flatten for global structural rules
        parts = self.flatten_unions(root_node)
        
        # 3. Apply Symmetry and Factoring
        parts = self.find_symmetries(parts)
        
        # 4. Final rebuild
        return self.rebuild_union(parts)

# --- EXECUTION ---
optimizer = ShapeOptimizer()
results = {}

# Example loop with improved error handling
for shape_id, data in shapes_l0.items():
    try:
        original_dsl = data.get("dsl")
        if original_dsl is None:
            continue

        optimized = optimizer.run(copy.deepcopy(original_dsl))
        results[shape_id] = optimized
        
        orig_count = len(optimizer.flatten_unions(original_dsl))
        opt_count = len(optimizer.flatten_unions(optimized))
        
        if opt_count < orig_count:
            print(f"✅ {shape_id}: Reduced from {orig_count} to {opt_count} nodes.")
            
    except Exception as e:
        logger.error(f"Critical failure on {shape_id}: {e}", exc_info=True)

INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (6 parts, axis AY)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AY)
INFO: Rule: SymTrans detected (4 parts, axis AX)


INFO: Rule: SymTrans detected (4 parts, axis AY)
INFO: Rule: SymTrans detected (6 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (9 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (5 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (5 parts, axis AX)
INFO: Rule: SymTrans detected (3 parts, axis AX)
INFO: Rule: SymTrans detected (3 parts, axis AX)
INFO: Rule: SymTrans detected (8 parts, axis AY)
INFO: Rule: SymTrans detected (6 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AY)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (6 parts, axis AX)
INFO: Rule: SymTrans detected (3 parts, axis AX)
INFO: Rule: SymTrans detected (5 parts, axis AX)
INFO: Rule: SymTrans detected (3 parts, axis AX)
INFO: Rule: SymTrans detected (7 parts, axis AX)
INFO: Rule: SymTrans

✅ 172_0_0: Reduced from 11 to 6 nodes.
✅ 173_0_1: Reduced from 6 to 4 nodes.
✅ 182_0_4: Reduced from 6 to 5 nodes.
✅ 186_0_5: Reduced from 9 to 7 nodes.
✅ 187_0_6: Reduced from 15 to 8 nodes.
✅ 197_0_7: Reduced from 16 to 14 nodes.
✅ 272_0_9: Reduced from 6 to 5 nodes.
✅ 278_0_10: Reduced from 11 to 6 nodes.
✅ 280_0_11: Reduced from 16 to 15 nodes.
✅ 285_0_12: Reduced from 9 to 7 nodes.
✅ 286_0_13: Reduced from 15 to 11 nodes.
✅ 314_0_15: Reduced from 11 to 6 nodes.
✅ 315_0_16: Reduced from 6 to 4 nodes.
✅ 328_0_17: Reduced from 16 to 14 nodes.
✅ 340_0_19: Reduced from 6 to 5 nodes.
✅ 341_0_20: Reduced from 6 to 5 nodes.
✅ 347_0_21: Reduced from 9 to 7 nodes.
✅ 349_0_22: Reduced from 15 to 12 nodes.
✅ 456_0_23: Reduced from 8 to 6 nodes.
✅ 470_0_24: Reduced from 7 to 6 nodes.
✅ 515_0_27: Reduced from 10 to 9 nodes.
✅ 518_0_29: Reduced from 8 to 7 nodes.
✅ 523_0_30: Reduced from 7 to 5 nodes.
✅ 526_0_31: Reduced from 14 to 12 nodes.
✅ 539_0_32: Reduced from 12 to 7 nodes.
✅ 543_0_33: Re

INFO: Rule: SymTrans detected (3 parts, axis AX)
INFO: Rule: SymTrans detected (3 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (3 parts, axis AY)
INFO: Rule: SymTrans detected (5 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (6 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (5 parts, axis AX)
INFO: Rule: SymTrans detected (3 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (3 parts, axis AX)
INFO: Rule: SymTrans detected (5 parts, axis AY)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (7 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (3 parts, axis AX)
INFO: Rule: SymTrans

✅ 2593_0_292: Reduced from 12 to 9 nodes.
✅ 2598_0_294: Reduced from 9 to 7 nodes.
✅ 2602_0_296: Reduced from 11 to 8 nodes.
✅ 2605_0_297: Reduced from 15 to 10 nodes.
✅ 2607_0_298: Reduced from 10 to 7 nodes.
✅ 2611_0_299: Reduced from 10 to 6 nodes.
✅ 2612_0_300: Reduced from 6 to 4 nodes.
✅ 2614_0_301: Reduced from 9 to 6 nodes.
✅ 2620_0_302: Reduced from 10 to 9 nodes.
✅ 2623_0_303: Reduced from 10 to 7 nodes.
✅ 2631_0_306: Reduced from 16 to 14 nodes.
✅ 2636_0_307: Reduced from 4 to 3 nodes.
✅ 2637_0_308: Reduced from 6 to 4 nodes.
✅ 2645_0_310: Reduced from 7 to 6 nodes.
✅ 2647_0_311: Reduced from 9 to 7 nodes.
✅ 2648_0_312: Reduced from 12 to 10 nodes.
✅ 2650_0_313: Reduced from 8 to 7 nodes.
✅ 2653_0_314: Reduced from 15 to 10 nodes.
✅ 2655_0_315: Reduced from 6 to 5 nodes.
✅ 2656_0_316: Reduced from 8 to 6 nodes.
✅ 2663_0_317: Reduced from 11 to 7 nodes.
✅ 2666_0_319: Reduced from 12 to 9 nodes.
✅ 2673_0_320: Reduced from 16 to 12 nodes.
✅ 2681_0_323: Reduced from 7 to 5 nodes

INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (6 parts, axis AY)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (8 parts, axis AY)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (8 parts, axis AY)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (6 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (8 parts, axis AY)
INFO: Rule: SymTrans detected (4 parts, axis AY)
INFO: Rule: SymTrans detected (4 parts, axis AY)
INFO: Rule: SymTrans

✅ 35103_0_584: Reduced from 15 to 12 nodes.
✅ 35104_0_585: Reduced from 6 to 4 nodes.
✅ 35111_0_587: Reduced from 16 to 12 nodes.
✅ 35119_0_589: Reduced from 11 to 6 nodes.
✅ 35121_0_590: Reduced from 9 to 7 nodes.
✅ 35123_0_591: Reduced from 15 to 13 nodes.
✅ 35131_0_592: Reduced from 11 to 6 nodes.
✅ 35142_0_593: Reduced from 9 to 7 nodes.
✅ 35144_0_594: Reduced from 14 to 12 nodes.
✅ 35146_0_596: Reduced from 9 to 7 nodes.
✅ 35153_0_597: Reduced from 11 to 6 nodes.
✅ 35178_0_598: Reduced from 11 to 6 nodes.
✅ 35180_0_599: Reduced from 12 to 11 nodes.
✅ 35184_0_600: Reduced from 12 to 10 nodes.
✅ 35186_0_602: Reduced from 15 to 10 nodes.
✅ 35189_0_604: Reduced from 11 to 6 nodes.
✅ 35197_0_607: Reduced from 15 to 13 nodes.
✅ 35200_0_608: Reduced from 6 to 4 nodes.
✅ 35201_0_609: Reduced from 15 to 8 nodes.
✅ 35203_0_610: Reduced from 15 to 13 nodes.
✅ 35204_0_611: Reduced from 6 to 4 nodes.
✅ 35205_0_612: Reduced from 9 to 7 nodes.
✅ 35213_0_613: Reduced from 15 to 13 nodes.
✅ 35222_

INFO: Rule: SymTrans detected (5 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (3 parts, axis AX)
INFO: Rule: SymTrans detected (6 parts, axis AY)
INFO: Rule: SymTrans detected (8 parts, axis AY)
INFO: Rule: SymTrans detected (6 parts, axis AX)
INFO: Rule: SymTrans detected (5 parts, axis AX)
INFO: Rule: SymTrans detected (5 parts, axis AX)
INFO: Rule: SymTrans detected (5 parts, axis AX)
INFO: Rule: SymTrans detected (5 parts, axis AX)
INFO: Rule: SymTrans detected (5 parts, axis AX)
INFO: Rule: SymTrans detected (5 parts, axis AX)
INFO: Rule: SymTrans detected (3 parts, axis AX)
INFO: Rule: SymTrans detected (6 parts, axis AX)
INFO: Rule: SymTrans detected (5 parts, axis AX)
INFO: Rule: SymTrans detected (8 parts, axis AY)
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Rule: SymTrans detected (5 parts, axis AX)
INFO: Rule: SymTrans detected (4 parts, axis AY)
INFO: Rule: SymTrans detected (3 parts, axis AX)
INFO: Rule: SymTrans

✅ 36201_0_852: Reduced from 15 to 7 nodes.
✅ 36202_0_853: Reduced from 6 to 4 nodes.
✅ 36203_0_854: Reduced from 14 to 13 nodes.
✅ 36209_0_855: Reduced from 11 to 5 nodes.
✅ 36212_0_857: Reduced from 10 to 7 nodes.
✅ 36213_0_858: Reduced from 11 to 10 nodes.
✅ 36217_0_859: Reduced from 6 to 5 nodes.
✅ 36220_0_861: Reduced from 8 to 6 nodes.
✅ 36225_0_862: Reduced from 9 to 8 nodes.
✅ 36228_0_863: Reduced from 10 to 9 nodes.
✅ 36229_0_864: Reduced from 10 to 8 nodes.
✅ 36236_0_867: Reduced from 11 to 6 nodes.
✅ 36239_0_868: Reduced from 6 to 4 nodes.
✅ 36245_0_870: Reduced from 14 to 9 nodes.
✅ 36248_0_872: Reduced from 10 to 7 nodes.
✅ 36249_0_873: Reduced from 6 to 5 nodes.
✅ 36251_0_874: Reduced from 9 to 8 nodes.
✅ 36255_0_875: Reduced from 7 to 5 nodes.
✅ 36256_0_876: Reduced from 10 to 8 nodes.
✅ 36258_0_878: Reduced from 12 to 10 nodes.
✅ 36267_0_879: Reduced from 14 to 12 nodes.
✅ 36269_0_880: Reduced from 8 to 5 nodes.
✅ 36277_0_882: Reduced from 15 to 11 nodes.
✅ 36284_0_884: 

In [10]:
import pandas as pd
import copy
import logging

# Helper to turn the node tree into a string for the CSV
def dsl_to_string(node):
    if node is None:
        return ""
    # This assumes your DSL nodes have a __repr__ or __str__ 
    # If not, we can use: str(type(node).__name__)
    return str(node)

stats_list = []

for shape_id, data in shapes_l0.items():
    try:
        original_dsl = data.get("dsl")
        if original_dsl is None:
            logger.warning(f"SKIP: {shape_id} - No DSL found.")
            continue

        # 1. Run Optimization
        # We keep a copy of the original for the 'before' string
        original_copy = copy.deepcopy(original_dsl)
        optimized = optimizer.run(copy.deepcopy(original_dsl))
        
        # 2. Extract Metrics
        orig_count = len(optimizer.flatten_unions(original_copy))
        opt_count = len(optimizer.flatten_unions(optimized))
        reduction = orig_count - opt_count
        
        # 3. Logging (No emojis)
        logger.info(f"Shape: {shape_id} | Reduction: {reduction}")
        
        # 4. Append to list for DataFrame
        stats_list.append({
            "shape_id": shape_id,
            "status": "SUCCESS",
            "original_count": orig_count,
            "optimized_count": opt_count,
            "reduction": reduction,
            "original_dsl": dsl_to_string(original_copy),
            "optimized_dsl": dsl_to_string(optimized)
        })
            
    except Exception as e:
        logger.error(f"FAILURE: {shape_id} - {str(e)}")
        stats_list.append({
            "shape_id": shape_id,
            "status": f"ERROR: {str(e)}",
            "original_count": None,
            "optimized_count": None,
            "reduction": None,
            "original_dsl": None,
            "optimized_dsl": None
        })

# --- Pandas Processing ---
df = pd.DataFrame(stats_list)

# Calculate percentage
df['reduction_pct'] = (df['reduction'] / df['original_count'] * 100).round(2).fillna(0)

# Reorder columns for better readability in CSV
# column_order = [
#     "shape_id", "status", "original_count", "optimized_count", 
#     "reduction", "reduction_pct", "original_dsl", "optimized_dsl"
# ]
# df = df[column_order]

# Save to CSV
# df.to_csv("shape_optimization_report.csv", index=False)

logger.info("CSV Report generated with 'original_dsl' and 'optimized_dsl' columns.")

INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Shape: 172_0_0 | Reduction: 5
INFO: Shape: 173_0_1 | Reduction: 2
INFO: Shape: 178_0_2 | Reduction: 0
INFO: Shape: 180_0_3 | Reduction: 0
INFO: Shape: 182_0_4 | Reduction: 1
INFO: Shape: 186_0_5 | Reduction: 2
INFO: Rule: SymTrans detected (6 parts, axis AY)
INFO: Shape: 187_0_6 | Reduction: 7
INFO: Shape: 197_0_7 | Reduction: 2
INFO: Shape: 257_0_8 | Reduction: 0
INFO: Shape: 272_0_9 | Reduction: 1
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Shape: 278_0_10 | Reduction: 5
INFO: Shape: 280_0_11 | Reduction: 1
INFO: Shape: 285_0_12 | Reduction: 2
INFO: Rule: SymTrans detected (4 parts, axis AY)
INFO: Shape: 286_0_13 | Reduction: 4
INFO: Shape: 308_0_14 | Reduction: 0
INFO: Rule: SymTrans detected (4 parts, axis AX)
INFO: Shape: 314_0_15 | Reduction: 5
INFO: Shape: 315_0_16 | Reduction: 2
INFO: Shape: 328_0_17 | Reduction: 2
INFO: Shape: 331_0_18 | Reduction: 0
INFO: Shape: 340_0_19 | Reduction: 1
INFO: Shape: 341_0_20 | R

In [11]:
from abstractionsshapecoder.dsl_utils import collect_singleton_and_pair_data
from abstractionsshapecoder.debug_utils import debug_info, debug_success

# 1. Isolate the DSL objects from the dictionary
all_l0_dsl_trees = [data['dsl'] for data in shapes_l0.values()]

debug_info(f"Extracting parameters from {len(all_l0_dsl_trees)} L0 shapes...")

# 2. Run the collection utility
# s_data_l0: { 'NodeName': [ [params], [params], ... ] }
# p_data_l0: { 'Parent(Child)': [ [params], [params], ... ] }
s_data_l0, p_data_l0 = collect_singleton_and_pair_data(all_l0_dsl_trees)

debug_success(f"Extraction complete.")
debug_info(f"Unique Singletons found: {len(s_data_l0)}")
debug_info(f"Unique Parent-Child Pairs found: {len(p_data_l0)}")

[INFO] Extracting parameters from 1100 L0 shapes...
[SUCCESS] Extraction complete.
[INFO] Unique Singletons found: 3
[INFO] Unique Parent-Child Pairs found: 5


In [12]:
import pandas as pd

# Create a summary for Singletons
singleton_summary = []
for node_type, param_list in s_data_l0.items():
    singleton_summary.append({
        "Node Type": node_type,
        "Total Occurrences": len(param_list),
        "Param Dimension": len(param_list[0]) if param_list else 0,
        "Example Params": param_list[0] if param_list else []
    })

# Create a summary for Pairs
pair_summary = []
for pair_name, param_list in p_data_l0.items():
    pair_summary.append({
        "Relationship": pair_name,
        "Total Occurrences": len(param_list),
        "Combined Dimension": len(param_list[0]) if param_list else 0
    })

# Convert to DataFrames for easy viewing
df_singletons = pd.DataFrame(singleton_summary).sort_values("Total Occurrences", ascending=False)
df_pairs = pd.DataFrame(pair_summary).sort_values("Total Occurrences", ascending=False)

print("=== L0 SINGLETON CENSUS ===")
display(df_singletons)

print("\n=== L0 PAIR CENSUS ===")
display(df_pairs)

=== L0 SINGLETON CENSUS ===


,Node Type,Total Occurrences,Param Dimension,Example Params
0,Cuboid,10268,3,"[0.68, 0.09, 0.66]"
2,Translate,10268,3,"[-0.0, -0.17, 0.0]"
1,Rotate,3416,4,"[-0.17410813759359595, 0.0, 0.0, 0.98472653890..."



=== L0 PAIR CENSUS ===


,Relationship,Total Occurrences,Combined Dimension
2,Translate(Cuboid),10268,6
4,Union(Translate),8656,3
0,Rotate(Rotate),1804,8
1,Rotate(Translate),1612,7
3,Union(Rotate),1612,4


In [13]:
results

{'172_0_0': <abstractionsshapecoder.dsl_nodes.Union at 0x1588f8b26f0>,
 '173_0_1': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd615af0>,
 '178_0_2': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd614b90>,
 '180_0_3': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd6163c0>,
 '182_0_4': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd614770>,
 '186_0_5': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd617b60>,
 '187_0_6': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd616c00>,
 '197_0_7': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd5eb1d0>,
 '257_0_8': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd616180>,
 '272_0_9': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd5e9d30>,
 '278_0_10': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd5ea840>,
 '280_0_11': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd6344d0>,
 '285_0_12': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd5eb500>,
 '286_0_13': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd635c40>,
 '

In [14]:
results

{'172_0_0': <abstractionsshapecoder.dsl_nodes.Union at 0x1588f8b26f0>,
 '173_0_1': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd615af0>,
 '178_0_2': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd614b90>,
 '180_0_3': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd6163c0>,
 '182_0_4': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd614770>,
 '186_0_5': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd617b60>,
 '187_0_6': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd616c00>,
 '197_0_7': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd5eb1d0>,
 '257_0_8': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd616180>,
 '272_0_9': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd5e9d30>,
 '278_0_10': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd5ea840>,
 '280_0_11': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd6344d0>,
 '285_0_12': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd5eb500>,
 '286_0_13': <abstractionsshapecoder.dsl_nodes.Union at 0x158cd635c40>,
 '